# DRL-NFV — Baseline Strategies (Kaggle)

Runs **GreedyFIFS**, **BestFit**, and **DeadlineAwareGreedy** VNF placement strategies from the [`tacongnam/DRL-NFV`](https://github.com/tacongnam/DRL-NFV) repo (branch `baseline-fixes`).

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────
!pip install -q networkx gymnasium matplotlib pandas numpy

## Clone the repo

In [ ]:
# ── 2. Clone repo (branch baseline-fixes) ────────────────────
import os

REPO_DIR = "/kaggle/working/DRL-NFV"

if not os.path.exists(REPO_DIR):
    !git clone -b baseline-fixes https://github.com/tacongnam/DRL-NFV.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

print("Done. Contents:", os.listdir(REPO_DIR))

In [ ]:
# ── 3. Add repo to Python path ────────────────────────────────
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("sys.path[0] =", sys.path[0])

In [ ]:
## Generate test data — all topologies / distributions / difficulties

In [ ]:
# ── 4. Generate test data for all scenarios ───────────────────
# Each (topology, distribution, difficulty) combo → its own sub-folder.
# cogent (197 nodes) is included but is much slower than nsf/conus.

import itertools

SCENARIOS = list(itertools.product(
    ["nsf", "conus", "cogent"],                     # topologies
    ["uniform", "rural", "urban", "centers"],       # distributions
    ["easy", "hard"],                               # difficulties
))

SCALE        = 50
NUM_REQUESTS = 100   # keep smaller for conus/cogent to avoid timeouts
NUM_FILES    = 2

DATA_ROOT = "/kaggle/working/data_test"
os.makedirs(DATA_ROOT, exist_ok=True)

for topology, distribution, difficulty in SCENARIOS:
    out_dir = f"{DATA_ROOT}/{topology}_{distribution}_{difficulty}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"Generating {topology}/{distribution}/{difficulty} ...", end=" ", flush=True)
    ret = os.system(
        f"python {REPO_DIR}/data/generate.py"
        f" --topology {topology}"
        f" --distribution {distribution}"
        f" --difficulty {difficulty}"
        f" --scale {SCALE}"
        f" --requests {NUM_REQUESTS}"
        f" --num-files {NUM_FILES}"
        f" --output {out_dir}"
        f" > /dev/null 2>&1"
    )
    status = "OK" if ret == 0 else f"FAILED (code {ret})"
    print(status)

print(f"\nAll done. {len(SCENARIOS)} scenarios generated.")

In [ ]:
## Run all 3 strategies across every scenario

# ── 5. Run baselines on every scenario, collect results ───────
import json, glob

BASELINES   = ["fifs", "bestfit", "deadline"]
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

all_rows = []

for topology, distribution, difficulty in SCENARIOS:
    scenario_key = f"{topology}/{distribution}/{difficulty}"
    test_dir     = f"{DATA_ROOT}/{topology}_{distribution}_{difficulty}"
    plot_out     = f"{RESULTS_DIR}/{topology}_{distribution}_{difficulty}.png"

    # Pick only first JSON file to keep runtime reasonable
    files = sorted(glob.glob(f"{test_dir}/*.json"))
    if not files:
        print(f"[SKIP] No data found for {scenario_key}")
        continue

    print(f"Running {scenario_key} on {len(files)} file(s)...", flush=True)

    # Run main.py — it prints stats to stdout and saves the plot
    ret = os.system(
        f"python {REPO_DIR}/main.py"
        f" --mode      baseline"
        f" --baselines {' '.join(BASELINES)}"
        f" --test-dir  {test_dir}"
        f" --plot-out  {plot_out}"
        f" > {RESULTS_DIR}/{topology}_{distribution}_{difficulty}.log 2>&1"
    )

    if ret != 0:
        print(f"  [WARN] non-zero exit for {scenario_key}")
        continue

    # Parse acceptance ratios from the log
    log_path = f"{RESULTS_DIR}/{topology}_{distribution}_{difficulty}.log"
    with open(log_path) as f:
        log = f.read()

    import re
    for line in log.splitlines():
        # Lines like: "GreedyFIFS                AR=0.429  ..."
        m = re.match(r"\s*(GreedyFIFS|BestFit|DeadlineAwareGreedy)\s+.*?(\d+\.\d+).*?(\d+)\s+(\d+).*?(\d+\.\d+)", line)
        if m:
            all_rows.append({
                "topology":     topology,
                "distribution": distribution,
                "difficulty":   difficulty,
                "strategy":     m.group(1),
                "ar":           float(m.group(2)),
            })

print(f"\nDone. Collected {len(all_rows)} strategy×scenario results.")

In [ ]:
## Results summary table & heatmap

In [ ]:
# ── 6. Summary table ──────────────────────────────────────────
import pandas as pd

if all_rows:
    df = pd.DataFrame(all_rows)
    pivot = df.pivot_table(
        index=["topology", "distribution", "difficulty"],
        columns="strategy",
        values="ar",
        aggfunc="mean",
    ).round(3)
    pivot.columns.name = None
    print(pivot.to_string())
    pivot.to_csv("/kaggle/working/baseline_summary.csv")
    print("\nSaved → /kaggle/working/baseline_summary.csv")
else:
    print("No results to display — check logs in", RESULTS_DIR)

In [ ]:
# ── 7. Heatmap: Acceptance Ratio per scenario × strategy ──────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

if all_rows:
    df = pd.DataFrame(all_rows)
    df["scenario"] = df["topology"] + "\n" + df["distribution"] + "\n" + df["difficulty"]

    strategies = ["GreedyFIFS", "BestFit", "DeadlineAwareGreedy"]
    pivot_heat = df.pivot_table(
        index="scenario", columns="strategy", values="ar", aggfunc="mean"
    ).reindex(columns=strategies)

    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_heat) * 0.45)))
    im = ax.imshow(pivot_heat.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label="Acceptance Ratio")

    ax.set_xticks(range(len(strategies)))
    ax.set_xticklabels(strategies, fontsize=10)
    ax.set_yticks(range(len(pivot_heat)))
    ax.set_yticklabels(pivot_heat.index, fontsize=8)

    for i in range(len(pivot_heat)):
        for j in range(len(strategies)):
            val = pivot_heat.values[i, j]
            if not pd.isna(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=8, color="black")

    ax.set_title("Acceptance Ratio — All Scenarios × Strategies", fontsize=13, fontweight="bold")
    plt.tight_layout()
    heatmap_path = "/kaggle/working/heatmap_all_scenarios.png"
    plt.savefig(heatmap_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved →", heatmap_path)
else:
    print("No results to plot.")